# LSTM / TCN trên PTB-XL: train + Membership Inference (TV1), bản Kaggle

Pipeline: PTB-XL → split member / non-member chung của nhóm → tune trên `val` → train **target** → train **8 shadow** → tấn công **Loss / Conf / Entropy, LiRA, RMIA, Temporal** → TPR @ 0,1% và 1% FPR + bootstrap CI → ghi `results/<model>/<task>/<attack>.json` vào `/kaggle/working`.

Một notebook cho cả 4 tổ hợp: chỉ đổi `MODEL` (`lstm` / `tcn`) và `TASK` (`cls` / `fcst`) ở cell cấu hình.

**Chuẩn bị (một lần)**
1. **Add Input → Datasets**, tìm `khyeh0719/ptb-xl-dataset` (PTB-XL v1.0.1). Kaggle đã giải nén sẵn, không cần zip.
2. Không cần upload `dataset/` của repo: notebook chứa sẵn bản sao `make_split.py`, tự tạo lại split từ PTB-XL trong input rồi so sha256 với `ptbxl_record_s42.meta.json` của repo (cũng nhúng sẵn). Lệch hash thì dừng.
3. **Settings → Accelerator → GPU T4 x2** (notebook dùng 1 GPU). Internet bật hay tắt đều được: không có `wfdb` thì notebook tự đọc file `.dat`.

**Chạy**
- Thử nhanh: chạy từng cell trong phiên tương tác.
- Chạy thật: **Save Version → Save & Run All (Commit)**. Notebook chạy nền tới khi xong (giới hạn 12 giờ mỗi phiên), output nằm ở tab *Output* của version.
- Nếu một version bị dừng giữa chừng (hết 12 giờ): **Add Input → Notebook Output**, chọn version trước của chính notebook này, rồi Save & Run All lại. Notebook chép `ckpt/` cũ vào và bỏ qua phần đã xong (cấu hình tune, target, từng shadow).

Thứ tự đề xuất: `tcn-cls` → `lstm-cls` → `tcn-fcst` → `lstm-fcst` (mỗi tổ hợp một version). Chi tiết: `report/BaoCao_TV1_LSTM_TCN.md` mục 4.

In [ ]:
try:                                       # PTB-XL is read with wfdb if available, otherwise with the built-in reader (Cell 3)
    import wfdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=False)   # fails quietly when Internet is off

## 1. Đường dẫn
PTB-XL được tìm trong `/kaggle/input`. Split chung của nhóm được tạo lại từ PTB-XL bằng chính `make_split.py` của repo (nhúng trong cell) vào `/tmp/tv1_dataset`, và phải khớp sha256 với bản trong repo. Checkpoint và kết quả ghi vào `/kaggle/working` (là output của version). Cache tín hiệu (~0,5 GB) để ở `/tmp` cho khỏi lọt vào output.

In [ ]:
import os, sys, glob, json, time, math, random, hashlib, subprocess, shutil
import numpy as np
KAGGLE_INPUT = os.environ.get('TV1_KAGGLE_INPUT', '/kaggle/input')
DRIVE_ROOT = os.environ.get('TV1_WORK', '/kaggle/working')          # ckpt/ and results/ become the notebook output
PTBXL_ZIP = None
LOCAL_CACHE = os.environ.get('TV1_CACHE', '/tmp/cache_tv1')
DATASET_DIR = os.environ.get('TV1_DATASET_DIR', '/tmp/tv1_dataset')  # make_split.py + splits/, rebuilt here
os.makedirs(f'{DATASET_DIR}/splits', exist_ok=True)

def find_input(name):
    """First file called `name` under /kaggle/input (skips the 40k-file PTB-XL record folders)."""
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        dirs[:] = sorted(d for d in dirs if d not in ('records100', 'records500'))
        if name in files:
            return os.path.join(root, name)

csv = find_input('ptbxl_database.csv')
assert csv, f'PTB-XL not found under {KAGGLE_INPUT}: Add Input -> Datasets -> khyeh0719/ptb-xl-dataset'
PTBXL_ROOT = os.path.dirname(csv)                                   # PTB-XL is already unpacked by Kaggle
print('PTB-XL:', PTBXL_ROOT)

# ---- the group split, rebuilt from PTB-XL with the repo's own code (no extra upload needed) ----
# verbatim copy of dataset/make_split.py
MAKE_SPLIT_SRC = '''"""Shared PTB-XL member / non-member split for all models (LSTM, TCN, PatchTST, iTransformer, MOMENT).

Uses only the 5-superclass, single-label records (same filter as the MOMENT tutorial), so every
team member gets identical ecg_id sets. Output keys follow the MOMENT dataset convention:

    train = member       (used to train the target model)
    test  = non-member   (never seen by the target model)
    val   = small held-out set carved out of the non-member pool (not used for the attack)

Usage:
    python make_split.py --ptbxl-dir /path/to/ptb-xl --level both --seed 42
"""
import argparse
import ast
import hashlib
import json
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

CLASSES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']   # sorted, same as sklearn LabelEncoder in momentfm


def load_single_label_records(ptbxl_dir):
    df = pd.read_csv(os.path.join(ptbxl_dir, 'ptbxl_database.csv'), index_col='ecg_id')
    codes = pd.read_csv(os.path.join(ptbxl_dir, 'scp_statements.csv'), index_col=0)
    diag = codes[codes.diagnostic == 1.0].diagnostic_class
    df['scp_codes'] = df.scp_codes.apply(ast.literal_eval)
    df['superclass'] = df.scp_codes.apply(lambda d: sorted({diag[k] for k in d if k in diag.index and pd.notna(diag[k])}))
    df = df[df.superclass.apply(len) == 1].copy()
    df['superclass'] = df.superclass.apply(lambda c: c[0])
    assert set(df.superclass) <= set(CLASSES), set(df.superclass)
    df['label'] = df.superclass.map({c: i for i, c in enumerate(CLASSES)})
    return df.sort_index()


def make_split(df, level, seed, n_val):
    ids = df.index.values
    if level == 'record':
        member, non = train_test_split(ids, test_size=0.5, stratify=df.label.values, random_state=seed)
    elif level == 'user':
        gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
        mi, ni = next(gss.split(ids, groups=df.patient_id.values))
        member, non = ids[mi], ids[ni]
        pid = df.patient_id
        assert not (set(pid.loc[member]) & set(pid.loc[non])), 'patient leak between member and non-member'
    else:
        raise ValueError(level)
    non = np.random.default_rng(seed).permutation(non)
    split = {'train': sorted(int(i) for i in member),
             'val': sorted(int(i) for i in non[:n_val]),
             'test': sorted(int(i) for i in non[n_val:])}
    assert not (set(split['train']) & set(split['val'])) and not (set(split['train']) & set(split['test']))
    assert not (set(split['val']) & set(split['test']))
    return split


def make_shadow_masks(df, level, seed, k):
    """Balanced in/out masks for LiRA / RMIA shadow models over the whole single-label pool.

    masks[j, i] = True  <=>  shadow j is trained on record ids[i]. Every unit (record, or patient at
    user level) is "in" for exactly k/2 shadows, so each sample has k/2 IN and k/2 OUT reference logits.
    Shadows use the same training recipe (epochs, optimizer, lr) as the target.
    """
    assert k % 2 == 0, 'k must be even'
    ids = df.index.values
    unit = ids if level == 'record' else df.patient_id.values
    units = np.unique(unit)
    rng = np.random.default_rng(seed + 1000)
    unit_masks = np.zeros((k, len(units)), dtype=bool)
    for u in range(len(units)):
        unit_masks[rng.permutation(k)[:k // 2], u] = True
    masks = unit_masks[:, np.searchsorted(units, unit)]
    ref_ids = np.sort(rng.choice(ids, size=min(5000, len(ids)), replace=False))   # RMIA population Z
    return ids, masks, ref_ids


def write_shadow(df, level, seed, k, out_dir):
    ids, masks, ref_ids = make_shadow_masks(df, level, seed, k)
    stem = f'ptbxl_{level}_s{seed}'
    path = os.path.join(out_dir, f'shadow_masks_{level}_K{k}_s{seed}.npz')
    np.savez_compressed(path, ids=ids, masks=masks, ref_ids=ref_ids)
    with open(path, 'rb') as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    meta_path = os.path.join(out_dir, stem + '.meta.json')
    with open(meta_path) as f:
        meta = json.load(f)
    meta['shadow'] = {'file': os.path.basename(path), 'K': k, 'sha256': digest, 'n_ref': int(len(ref_ids)),
                      'in_fraction': float(masks.mean()),
                      'rmia_defaults': {'gamma': 2.0, 'a': 0.3, 'offline': True}}
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=1)
    print(' ', os.path.basename(path), masks.shape, 'sha256', digest[:12])


def write(df, split, level, seed, n_val, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    stem = f'ptbxl_{level}_s{seed}'
    path = os.path.join(out_dir, stem + '.json')
    with open(path, 'w') as f:
        json.dump(split, f)
    with open(path, 'rb') as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    meta = {
        'level': level, 'seed': seed, 'n_val': n_val, 'classes': CLASSES, 'sha256': digest,
        'n_records_total': int(len(df)), 'n_patients_total': int(df.patient_id.nunique()),
        'sizes': {k: len(v) for k, v in split.items()},
        'class_counts': {k: np.bincount(df.label.loc[v].values, minlength=len(CLASSES)).tolist() for k, v in split.items()},
        'n_patients': {k: int(df.patient_id.loc[v].nunique()) for k, v in split.items()},
    }
    with open(os.path.join(out_dir, stem + '.meta.json'), 'w') as f:
        json.dump(meta, f, indent=1)
    print(stem, meta['sizes'], 'sha256', digest[:12])
    if meta['sizes']['test'] < 10000:
        print(f"  WARNING: only {meta['sizes']['test']} non-members; TPR@0.1%FPR rests on ~{int(meta['sizes']['test'] * 0.001)} samples")


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--ptbxl-dir', required=True, help='folder containing ptbxl_database.csv')
    ap.add_argument('--level', choices=['record', 'user', 'both'], default='both')
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--n-val', type=int, default=500)
    ap.add_argument('--out-dir', default=os.path.join(os.path.dirname(os.path.abspath(__file__)), 'splits'))
    ap.add_argument('--shadow-k', type=int, default=8, help='number of LiRA/RMIA shadow masks (even); 0 to skip')
    a = ap.parse_args()
    data = load_single_label_records(a.ptbxl_dir)
    for lv in (['record', 'user'] if a.level == 'both' else [a.level]):
        write(data, make_split(data, lv, a.seed, a.n_val), lv, a.seed, a.n_val, a.out_dir)
        if a.shadow_k:
            write_shadow(data, lv, a.seed, a.shadow_k, a.out_dir)
'''
# verbatim copy of dataset/splits/ptbxl_record_s42.meta.json (sha256 of the official split + shadow masks)
SPLIT_META = json.loads('''{
 "level": "record",
 "seed": 42,
 "n_val": 500,
 "classes": [
  "CD",
  "HYP",
  "MI",
  "NORM",
  "STTC"
 ],
 "sha256": "35dd8ca57797bed344db4bc8ff0f99b6f05e78add6c565bed54932d5687adce7",
 "n_records_total": 16272,
 "n_patients_total": 14641,
 "sizes": {
  "train": 8136,
  "val": 500,
  "test": 7636
 },
 "class_counts": {
  "train": [
   854,
   268,
   1269,
   4542,
   1203
  ],
  "val": [
   54,
   24,
   91,
   266,
   65
  ],
  "test": [
   801,
   244,
   1178,
   4275,
   1138
  ]
 },
 "n_patients": {
  "train": 7696,
  "val": 497,
  "test": 7211
 },
 "shadow": {
  "file": "shadow_masks_record_K8_s42.npz",
  "K": 8,
  "sha256": "fbda57864ba16e9d50fd1bcb1c1c5287548511dc476a02924d494c36aa2a5dc9",
  "n_ref": 5000,
  "in_fraction": 0.5,
  "rmia_defaults": {
   "gamma": 2.0,
   "a": 0.3,
   "offline": true
  }
 }
}''')
MASK_CONTENT_SHA256 = 'e3b5d6ccea6ee815f60e577b51a978b9928db6bcfe790e8251a01eee01795faa'   # sha256 of ids | masks | ref_ids

open(f'{DATASET_DIR}/make_split.py', 'w').write(MAKE_SPLIT_SRC)
sys.path.insert(0, DATASET_DIR)
import make_split

split_file = f'{DATASET_DIR}/splits/ptbxl_record_s42.json'
mask_file = f'{DATASET_DIR}/splits/shadow_masks_record_K8_s42.npz'
df_split = make_split.load_single_label_records(PTBXL_ROOT)
with open(split_file, 'w') as f:                                    # same bytes as make_split.write()
    json.dump(make_split.make_split(df_split, 'record', SPLIT_META['seed'], SPLIT_META['n_val']), f)
sha = hashlib.sha256(open(split_file, 'rb').read()).hexdigest()
assert sha == SPLIT_META['sha256'], f'rebuilt split sha256 {sha[:12]} != repo {SPLIT_META["sha256"][:12]}: different PTB-XL copy or sklearn version, stop'

ids, masks, ref_ids = make_split.make_shadow_masks(df_split, 'record', SPLIT_META['seed'], SPLIT_META['shadow']['K'])
h = hashlib.sha256(ids.astype('<i8').tobytes() + masks.astype(bool).tobytes(order='F') + ref_ids.astype('<i8').tobytes()).hexdigest()
assert h == MASK_CONTENT_SHA256, f'rebuilt shadow masks {h[:12]} differ from the repo ones, stop'
np.savez_compressed(mask_file, ids=ids, masks=masks, ref_ids=ref_ids)   # file bytes differ (zip timestamps), content is identical
json.dump(SPLIT_META, open(f'{DATASET_DIR}/splits/ptbxl_record_s42.meta.json', 'w'), indent=1)
del df_split, ids, masks, ref_ids
print('split rebuilt and matches the repo: sha256', sha[:12], SPLIT_META['sizes'], '->', DATASET_DIR, sorted(os.listdir(f'{DATASET_DIR}/splits')))

## 2. Cấu hình
Chỉ đổi `MODEL`, `TASK`. `EPOCHS` / `MAX_LR` sẽ bị ghi đè bằng cấu hình tune tốt nhất ở bước 6 (đặt `RUN_TUNE = False` nếu muốn tự cố định). Recipe còn lại (batch 16, AdamW, OneCycleLR, grad clip 1.0, K = 8 shadow) là bắt buộc theo `dataset/README.md`.

In [ ]:
MODEL = os.environ.get('TV1_MODEL', 'lstm')               # 'lstm' or 'tcn'
TASK = os.environ.get('TV1_TASK', 'cls')                  # 'cls' or 'fcst'
EPOCHS = int(os.environ.get('TV1_EPOCHS', 30))            # overwritten by the best tuning config in Cell 6
MAX_LR = float(os.environ.get('TV1_MAX_LR', 1e-3))
RUN_TUNE = os.environ.get('TV1_RUN_TUNE', '1') == '1'     # False: skip Cell 6 and keep EPOCHS / MAX_LR above
TUNE_GRID = [(20, 1e-3), (20, 3e-3), (50, 1e-3), (50, 3e-3), (100, 1e-3), (100, 3e-3)]   # <= 6 configs
if os.environ.get('TV1_TUNE_GRID'):                       # e.g. '2:1e-3,3:3e-3' for a quick test
    TUNE_GRID = [(int(e), float(l)) for e, l in (c.split(':') for c in os.environ['TV1_TUNE_GRID'].split(','))]

SEED, LEVEL, BATCH, N_BOOT = 42, 'record', 16, 1000
LOOKBACK, HORIZON = 512, 96
SEQ_LEN = LOOKBACK if TASK == 'cls' else LOOKBACK + HORIZON   # same seq_len as the MOMENT notebooks
K = 8                                                     # masks are balanced over all 8 shadows only

EVAL_AT = sorted({math.ceil(EPOCHS * f) for f in (0.1, 0.2, 0.5, 1.0)})   # 10/20/50/100% of EPOCHS
USE_BF16 = False                                          # fp32 is allowed on T4; set True on A100/L4
CLASSES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
MASK_PATH = f'{DATASET_DIR}/splits/shadow_masks_{LEVEL}_K8_s{SEED}.npz'
CKPT_DIR = f'{DRIVE_ROOT}/ckpt/{MODEL}_{TASK}'
RESULTS_DIR = f'{DRIVE_ROOT}/results/{MODEL}/{TASK}'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(LOCAL_CACHE, exist_ok=True)
print(MODEL, TASK, 'epochs', EPOCHS, 'max_lr', MAX_LR, 'eval_at', EVAL_AT, 'seq_len', SEQ_LEN)

### Chạy tiếp từ version trước (nếu có)
Nếu đã Add Input output của một version trước, chép `ckpt/<model>_<task>/` vào `/kaggle/working`. File đã có thì giữ nguyên.

In [ ]:
for prev in glob.glob(f'{KAGGLE_INPUT}/**/ckpt/{MODEL}_{TASK}', recursive=True):
    n = 0
    for f in os.listdir(prev):
        if not os.path.exists(f'{CKPT_DIR}/{f}'):
            shutil.copy(os.path.join(prev, f), CKPT_DIR); n += 1
    print('resumed', n, 'files from', prev)
print('ckpt now:', sorted(os.listdir(CKPT_DIR)))

## 3. Dữ liệu PTB-XL
Kiểm sha256 của split, đọc 16.272 bản ghi single-label (100 Hz, 12 lead), nội suy về `SEQ_LEN` (cls 512, fcst 608) và z-score theo lead. Lần đầu mất vài phút, sau đó đọc từ cache. Nếu một `assert` báo lỗi thì **dừng**: sai bản dataset hoặc sai split.

In [ ]:
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from scipy.interpolate import interp1d
try:
    import wfdb
except ImportError:
    wfdb = None
    print('wfdb not installed (Internet off?): using the built-in WFDB format-16 reader')

def read_record(path):
    """[n_samples, n_leads] in physical units. Uses wfdb when available; otherwise parses the .hea header and the
    format-16 .dat file (all PTB-XL records). The later z-score makes gain/baseline irrelevant, but they are applied."""
    if wfdb is not None:
        return wfdb.rdsamp(path)[0]
    lines = [l for l in open(path + '.hea').read().splitlines() if l.strip() and not l.startswith('#')]
    n_sig = int(lines[0].split()[1])
    gains, bases = [], []
    for l in lines[1:1 + n_sig]:
        tok = l.split()
        assert tok[1] == '16', f'unsupported WFDB format {tok[1]} in {path}.hea: pip install wfdb'
        g = tok[2].split('/')[0]
        gain, base = (float(g.split('(')[0]), int(g.split('(')[1].rstrip(')'))) if '(' in g else (float(g), int(tok[4]))
        gains.append(gain or 200.0); bases.append(base)
    raw = np.fromfile(os.path.join(os.path.dirname(path), lines[1].split()[0]), dtype='<i2').reshape(-1, n_sig)
    return (raw - np.array(bases)) / np.array(gains)

def control_randomness(seed=SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

if not glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    assert os.path.exists(PTBXL_ZIP), f'missing {PTBXL_ZIP}'
    # only the 100 Hz records and the two CSVs are needed (skips records500, most of the 3.2 GB)
    subprocess.run(['unzip', '-q', '-o', PTBXL_ZIP, '*ptbxl_database.csv', '*scp_statements.csv', '*records100/*', '-d', PTBXL_ROOT], check=True)
PTBXL_DIR = os.path.dirname(glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True)[0])

split = json.load(open(SPLIT_PATH))
meta = json.load(open(SPLIT_PATH.replace('.json', '.meta.json')))
sha = hashlib.sha256(open(SPLIT_PATH, 'rb').read()).hexdigest()
assert sha == meta['sha256'], 'split file differs from .meta.json: stop (README section 1)'
print({k: len(v) for k, v in split.items()}, 'sha256', sha[:12])

sys.path.insert(0, DATASET_DIR)
from make_split import load_single_label_records         # same label filter as the split
df = load_single_label_records(PTBXL_DIR)
mk = np.load(MASK_PATH)
pool_ids = mk['ids']                                      # every single-label record, sorted ecg_id
assert len(pool_ids) == len(df) and (pool_ids == df.index.values).all(), 'PTB-XL copy differs from the one used for the split'
pool_y = df.label.loc[pool_ids].values.astype(np.int64)

def load_signals(ids, seq_len):
    """Same preprocessing as momentfm PTBXL_dataset: whole 10 s record (1000 pts @100 Hz) linearly
    interpolated to seq_len, then z-score per lead per record."""
    cache = f'{LOCAL_CACHE}/ptbxl_{seq_len}_{hashlib.sha1(np.asarray(ids).tobytes()).hexdigest()[:10]}.npy'
    if os.path.exists(cache):
        return np.load(cache)
    out = np.empty((len(ids), 12, seq_len), dtype=np.float32)
    for i, e in enumerate(ids):
        sig = read_record(os.path.join(PTBXL_DIR, df.filename_lr.loc[e]))    # [1000, 12]
        x = sig.T.astype(np.float64)
        x = interp1d(np.linspace(0, 1, x.shape[1]), x, axis=-1)(np.linspace(0, 1, seq_len))
        out[i] = (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-8)
        if i % 2000 == 0: print('read', i, '/', len(ids))
    np.save(cache, out)
    return out

X_pool = torch.from_numpy(load_signals(pool_ids, SEQ_LEN))          # [N, 12, SEQ_LEN]
idx_m = np.flatnonzero(np.isin(pool_ids, split['train']))           # members
idx_v = np.flatnonzero(np.isin(pool_ids, split['val']))             # tuning only
idx_n = np.flatnonzero(np.isin(pool_ids, split['test']))            # non-members
assert len(idx_m) == meta['sizes']['train'] and len(idx_n) == meta['sizes']['test']
masks_pool = mk['masks'][:K]                                        # (K, N) True = shadow k trains on it
assert (masks_pool.sum(0) == K // 2).all(), 'every record needs K/2 IN and K/2 OUT shadows: keep K = 8'
ref_pool = np.flatnonzero(np.isin(pool_ids, mk['ref_ids']))
print('pool', X_pool.shape, 'member', len(idx_m), 'val', len(idx_v), 'non-member', len(idx_n))

## 4. Model
LSTM 2 lớp × 128 (hidden state cuối → Linear). TCN 7 block, 64 kênh, kernel 5, dilation 1…64 (receptive field 1017).

In [ ]:
from torch.nn.utils.parametrizations import weight_norm
OUT_DIM = 5 if TASK == 'cls' else 12 * HORIZON

class LSTMNet(nn.Module):
    def __init__(self, hidden=128, layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(12, hidden, layers, batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden, OUT_DIM)
    def forward(self, x):                              # x: [B, 12, T]
        h, _ = self.lstm(x.transpose(1, 2))            # [B, T, hidden]
        return self.head(h[:, -1])                     # last hidden state

class TemporalBlock(nn.Module):
    def __init__(self, c_in, c_out, k, d, dropout):
        super().__init__()
        pad = (k - 1) * d
        self.net = nn.Sequential(
            weight_norm(nn.Conv1d(c_in, c_out, k, padding=pad, dilation=d)), Chomp(pad), nn.ReLU(), nn.Dropout(dropout),
            weight_norm(nn.Conv1d(c_out, c_out, k, padding=pad, dilation=d)), Chomp(pad), nn.ReLU(), nn.Dropout(dropout))
        self.down = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()
    def forward(self, x):
        return F.relu(self.net(x) + self.down(x))

class Chomp(nn.Module):                                # drop the right padding -> causal convolution
    def __init__(self, n): super().__init__(); self.n = n
    def forward(self, x): return x[..., :-self.n] if self.n else x

class TCNNet(nn.Module):
    # receptive field = 1 + 2 (k-1)(2^levels - 1) = 1017 >= 608 with k=5, 7 levels
    def __init__(self, channels=64, levels=7, k=5, dropout=0.1):
        super().__init__()
        self.tcn = nn.Sequential(*[TemporalBlock(12 if i == 0 else channels, channels, k, 2 ** i, dropout) for i in range(levels)])
        self.head = nn.Linear(channels, OUT_DIM)
    def forward(self, x):                              # x: [B, 12, T]
        return self.head(self.tcn(x)[..., -1])         # last time step sees the whole input

def build_model():
    return {'lstm': LSTMNet, 'tcn': TCNNet}[MODEL]()

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
amp = lambda: torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16 and dev == 'cuda')
print(MODEL, 'params', sum(p.numel() for p in build_model().parameters()))

## 5. Recipe huấn luyện chung

In [ ]:
def inputs_targets(xb):
    if TASK == 'cls':
        return xb, None
    return xb[..., :LOOKBACK], xb[..., LOOKBACK:]

def forward(model, xb):
    out = model(xb).float()
    return out if TASK == 'cls' else out.view(-1, 12, HORIZON)

def batch_loss(model, xb, yb):
    x_in, tgt = inputs_targets(xb)
    out = forward(model, x_in)
    return F.cross_entropy(out, yb) if TASK == 'cls' else F.mse_loss(out, tgt)

@torch.no_grad()
def outputs(model, idx, bs=256):
    """cls: logits [n, 5]. fcst: predictions [n, 12, H]."""
    model.eval()
    res = []
    for s in range(0, len(idx), bs):
        x_in, _ = inputs_targets(X_pool[idx[s:s + bs]].to(dev))
        with amp():
            res.append(forward(model, x_in).cpu())
    return torch.cat(res)

def train(train_idx, epochs, max_lr, seed, on_epoch=None):
    control_randomness(seed)
    model = build_model().to(dev)
    steps = len(train_idx) // BATCH                                   # drop_last
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr / 25)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=epochs * steps)
    g = torch.Generator().manual_seed(seed)
    y_all = torch.from_numpy(pool_y)
    for ep in range(1, epochs + 1):
        model.train(); losses = []
        perm = torch.as_tensor(train_idx)[torch.randperm(len(train_idx), generator=g)]
        for s in range(steps):
            b = perm[s * BATCH:(s + 1) * BATCH]
            opt.zero_grad(set_to_none=True)
            with amp():
                loss = batch_loss(model, X_pool[b].to(dev), y_all[b].to(dev))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); losses.append(loss.item())
        if on_epoch: on_epoch(model, ep, float(np.mean(losses)))
    return model

def per_record_mse(pred, idx):
    return ((pred.double() - X_pool[idx][..., LOOKBACK:].double()) ** 2).mean(dim=(1, 2)).numpy()

def utility(out_m, out_n):
    if TASK == 'cls':
        a_m = float((out_m.argmax(1).numpy() == pool_y[idx_m]).mean()); a_n = float((out_n.argmax(1).numpy() == pool_y[idx_n]).mean())
        return {'train_acc': a_m, 'test_acc': a_n, 'gap': a_m - a_n}
    e_m, e_n = float(per_record_mse(out_m, idx_m).mean()), float(per_record_mse(out_n, idx_n).mean())
    return {'train_mse': e_m, 'test_mse': e_n, 'gap': e_n - e_m}

## 6. Tune trên `val` (≤ 6 cấu hình)
Chỉ nhìn utility trên `val`, không nhìn điểm MIA. Kết quả lưu ở `tuning.json`; chạy lại thì cấu hình đã xong được đọc lại. Cấu hình tốt nhất được dùng cho target và shadow.

In [ ]:
def tune():
    path = f'{CKPT_DIR}/tuning.json'
    rows = json.load(open(path)) if os.path.exists(path) else []
    done = {(r['epochs'], r['max_lr']) for r in rows}
    for ep, lr in TUNE_GRID:
        if (ep, lr) in done: continue
        t0 = time.time()
        model = train(idx_m, ep, lr, SEED)
        o = outputs(model, idx_v)
        val = float((o.argmax(1).numpy() == pool_y[idx_v]).mean()) if TASK == 'cls' else float(per_record_mse(o, idx_v).mean())
        rows.append({'epochs': ep, 'max_lr': lr, ('val_acc' if TASK == 'cls' else 'val_mse'): val, 'minutes': round((time.time() - t0) / 60, 1)})
        json.dump(rows, open(path, 'w'), indent=1); print(rows[-1])
    return rows

if RUN_TUNE:                    # finished configs are read back from tuning.json, so rerunning is cheap
    tuning_rows = tune()
    metric = 'val_acc' if TASK == 'cls' else 'val_mse'
    in_grid = [r for r in tuning_rows if (r['epochs'], r['max_lr']) in TUNE_GRID]
    best = (max if TASK == 'cls' else min)(in_grid, key=lambda r: r[metric])
    EPOCHS, MAX_LR = best['epochs'], best['max_lr']
    EVAL_AT = sorted({math.ceil(EPOCHS * f) for f in (0.1, 0.2, 0.5, 1.0)})
    print('best', best, '-> EPOCHS', EPOCHS, 'MAX_LR', MAX_LR, 'eval_at', EVAL_AT)

## 7. Target model
Train trên member, lưu output (logit hoặc dự đoán) ở các epoch 10 / 20 / 50 / 100%.

In [ ]:
TAG = f'{MODEL}_{TASK}_{LEVEL}_s{SEED}'
KEY = f"{MODEL}|{TASK}|{SEQ_LEN}|{EPOCHS}|{MAX_LR}|{BATCH}|{meta['sha256'][:12]}"

def run_target():
    done_path = f'{CKPT_DIR}/{TAG}.done.json'
    if os.path.exists(done_path) and json.load(open(done_path))['key'] == KEY:
        print('target already trained'); return json.load(open(done_path))['history']
    history = []
    def on_epoch(model, ep, train_loss):
        if ep not in EVAL_AT: return
        om, on = outputs(model, idx_m), outputs(model, idx_n)
        torch.save({'key': KEY, 'm': om, 'n': on}, f'{CKPT_DIR}/{TAG}_out_ep{ep}.pt')
        history.append({'epoch': ep, 'train_loss': train_loss, **utility(om, on)}); print(history[-1])
    t0 = time.time()
    train(idx_m, EPOCHS, MAX_LR, SEED, on_epoch)
    json.dump({'key': KEY, 'history': history, 'minutes': (time.time() - t0) / 60}, open(done_path, 'w'))
    return history
history = run_target()
target = torch.load(f'{CKPT_DIR}/{TAG}_out_ep{EPOCHS}.pt')
UTIL = utility(target['m'], target['n'])

## 8. 8 shadow model
Cùng recipe với target, train trên các bản ghi có mask = True. Shadow đã xong được bỏ qua.

In [ ]:
def run_shadow(j):
    path = f'{CKPT_DIR}/shadow_{TAG}_k{j}.pt'
    key = f"{KEY}|shadow{j}|{meta['shadow']['sha256'][:12]}"
    if os.path.exists(path) and torch.load(path)['key'] == key:
        return
    t0 = time.time()
    model = train(np.flatnonzero(masks_pool[j]), EPOCHS, MAX_LR, SEED + 1 + j)
    o = outputs(model, np.arange(len(pool_ids)))
    stat = o if TASK == 'cls' else torch.from_numpy(per_record_mse(o, np.arange(len(pool_ids))))   # logits or per-record MSE
    torch.save({'key': key, 'stat': stat}, path)
    print(f'shadow {j} done in {(time.time() - t0) / 60:.1f} min')
for j in range(K):
    run_shadow(j)

## 9. Metric và ghi JSON
Schema theo `results/_sample.json`.

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc
def tpr_at(fpr, tpr, t):
    m = fpr <= t
    return float(tpr[m].max()) if m.any() else 0.0
def mia_metrics(sm, sn):
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    return tpr_at(fpr, tpr, 1e-3), tpr_at(fpr, tpr, 1e-2), float(sk_auc(fpr, tpr))
def bootstrap_ci(sm, sn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    res = np.array([mia_metrics(sm[rng.integers(0, len(sm), len(sm))], sn[rng.integers(0, len(sn), len(sn))])[:2] for _ in range(n)])
    lo, hi = np.percentile(res, [2.5, 97.5], axis=0)
    return [float(lo[0]), float(hi[0])], [float(lo[1]), float(hi[1])]
def sig(v):                     # 9 significant digits, keeps tiny scores such as -CE = -1e-12 distinct
    return [float(f'{x:.9g}') for x in np.asarray(v, dtype=np.float64)]
TUNING = json.load(open(f'{CKPT_DIR}/tuning.json')) if os.path.exists(f'{CKPT_DIR}/tuning.json') else 'not run'
def save_result(attack, sm, sn, n_shadow=None, **extra):
    t01, t1, au = mia_metrics(sm, sn); c01, c1 = bootstrap_ci(sm, sn)
    out = {'model': MODEL, 'task': TASK, 'attack': attack, 'owner_tv': 1, 'level': LEVEL, 'dataset': 'ptbxl', 'seed': SEED,
           'n_member': int(len(sm)), 'n_nonmember': int(len(sn)),
           'scores_member': sig(sm), 'scores_nonmember': sig(sn),
           'tpr_at_0.1fpr': t01, 'tpr_at_1fpr': t1, 'auc': au,
           'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
           'utility': UTIL, 'n_shadow': n_shadow,
           'extra': {'epochs': EPOCHS, 'eval_epoch': EPOCHS, 'eval_epochs': EVAL_AT, 'batch': BATCH, 'max_lr': MAX_LR,
                     'backbone': repr(build_model()).split('(')[0], 'seq_len': SEQ_LEN, 'precision': 'bf16' if USE_BF16 else 'fp32',
                     'tuning': TUNING, 'split_sha256': meta['sha256'], 'shadow_masks_sha256': meta['shadow']['sha256'],
                     **({'lookback': LOOKBACK, 'horizon': HORIZON} if TASK == 'fcst' else {}), **extra}}
    json.dump(out, open(f'{RESULTS_DIR}/{attack}.json', 'w'))
    print(f'{attack:13s} TPR@0.1% {t01:.4f} {np.round(c01, 4)}  TPR@1% {t1:.4f} {np.round(c1, 4)}  AUC {au:.4f}')

## 10. Loss / Confidence / Entropy
Score càng cao càng giống member. fcst chỉ có Loss = `-MSE`.

In [ ]:
n_m = len(idx_m)
if TASK == 'cls':
    def log_p(z, ref):
        """log softmax of class `ref` as -log1p(sum_{j != ref} exp(z_j - z_ref)) in float64. A float32 log_softmax
        rounds every confident record to exactly 0 (p = 1), which ties members and non-members at the top score."""
        gaps = (z - z.gather(1, ref)).scatter(1, ref, float('-inf'))
        return -torch.log1p(gaps.exp().sum(1))
    def basic_scores(logits, y):
        z = logits.double(); top = z.argmax(1, keepdim=True)
        logp_max = log_p(z, top)
        logp = (z - z.gather(1, top)) + logp_max[:, None]          # log p_j, exact also for the tiny p_j
        return {'loss': log_p(z, torch.as_tensor(y).long()[:, None]).numpy(),   # -CE
                'conf': logp_max.numpy(),                                       # log max softmax: same ranking as max softmax
                'entropy': (logp.exp() * logp).sum(1).numpy()}                  # -H(p)
    sm, sn = basic_scores(target['m'], pool_y[idx_m]), basic_scores(target['n'], pool_y[idx_n])
else:
    sm, sn = {'loss': -per_record_mse(target['m'], idx_m)}, {'loss': -per_record_mse(target['n'], idx_n)}
SCORE_DESC = {'loss': '-CE (float64, log1p)', 'conf': 'log max softmax (float64, log1p; same ranking as max softmax)',
              'entropy': '-H(p) (float64, log1p)'} if TASK == 'cls' else {'loss': '-MSE'}
for a in sm:
    save_result(a, sm[a], sn[a], score=SCORE_DESC[a])
    top = max(sm[a].max(), sn[a].max())
    print(f'  {a}: at the top score {top:.3g}: {np.mean(sm[a] == top):.2%} of members, {np.mean(sn[a] == top):.2%} of non-members')
assert mia_metrics(sm['loss'], sn['loss'])[2] > 0.5, 'Loss AUC <= 0.5: check the score direction'

## 11. LiRA và RMIA
LiRA online / offline (phương sai toàn cục), RMIA offline / online (γ = 2, a = 0,3). Số chính: `lira` (online) và `rmia` (offline).

In [ ]:
from scipy.stats import norm
S = torch.stack([torch.load(f'{CKPT_DIR}/shadow_{TAG}_k{j}.pt')['stat'] for j in range(K)])
Min = masks_pool.astype(float); Mout = 1 - Min
idx_t = np.r_[idx_m, idx_n]
RMIA_A, RMIA_GAMMA = meta['shadow']['rmia_defaults']['a'], meta['shadow']['rmia_defaults']['gamma']

if TASK == 'cls':
    def phi_p(logits, y):
        z = logits.double(); y = torch.as_tensor(y).long()[:, None]
        phi = z.gather(1, y).squeeze(1) - torch.logsumexp(z.scatter(1, y, float('-inf')), 1)
        return phi.numpy(), torch.sigmoid(phi).numpy()
    PHI_S, P_S = map(np.stack, zip(*[phi_p(S[j], pool_y) for j in range(K)]))
    T = torch.full((len(pool_ids), 5), float('nan')); T[idx_m], T[idx_n] = target['m'], target['n']
    PHI_T, P_T = phi_p(T, pool_y)
else:
    S_mse = S.numpy().astype(np.float64)
    T_mse = np.full(len(pool_ids), np.nan)
    T_mse[idx_m], T_mse[idx_n] = per_record_mse(target['m'], idx_m), per_record_mse(target['n'], idx_n)
    PHI_S, PHI_T = -np.log(S_mse + 1e-8), -np.log(T_mse + 1e-8)
    TAU0 = float(np.median(S_mse[Mout.astype(bool)]))
    P_S, P_T = np.exp(-S_mse / TAU0), np.exp(-T_mse / TAU0)

def group_stats(W, X):
    c = W.sum(0); mu = (W * X).sum(0) / c
    return mu, (W * (X - mu) ** 2).sum(0) / (c - 1)
def lira(online):
    mu_in, v_in = group_stats(Min, PHI_S); mu_out, v_out = group_stats(Mout, PHI_S)
    s_in, s_out = np.sqrt(v_in[idx_t].mean()), np.sqrt(v_out[idx_t].mean())
    x = PHI_T[idx_t]
    if online:
        return norm.logpdf(x, mu_in[idx_t], s_in) - norm.logpdf(x, mu_out[idx_t], s_out)
    return (x - mu_out[idx_t]) / s_out
def rmia(online):
    p_in = (Min * P_S).sum(0) / Min.sum(0); p_out = (Mout * P_S).sum(0) / Mout.sum(0)
    p_bar = 0.5 * (p_in + p_out) if online else 0.5 * ((1 + RMIA_A) * p_out + (1 - RMIA_A))
    ratio = P_T / p_bar
    z = ratio[np.intersect1d(ref_pool, idx_t)]
    return np.searchsorted(np.sort(z), ratio[idx_t] / RMIA_GAMMA, side='right') / len(z)

fc = {'tau': TAU0} if TASK == 'fcst' else {}
r_off = rmia(False); print(f'RMIA offline: {np.mean(r_off == 0):.1%} scores = 0, {np.mean(r_off == 1):.1%} = 1 (near 100% at 0 means gamma saturates)')
for a, s in {'lira': lira(True), 'lira_offline': lira(False), 'rmia': rmia(False), 'rmia_online': rmia(True)}.items():
    save_result(a, s[:n_m], s[n_m:], n_shadow=K, **fc)

## 12. Temporal (gray-box)
cls: quỹ đạo CE / entropy / margin qua 4 epoch đánh giá. fcst: DFT 8 dải + trend + drift + log-MSE của residual. HistGradientBoosting, cross-fitting 5-fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
def cross_fit(X, y):
    score = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y):
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED).fit(X[tr], y[tr])
        score[te] = clf.predict_proba(X[te])[:, 1]
    return score
def slope(v, t):
    t = np.asarray(t, dtype=np.float64); t = t - t.mean()
    return (v * t).sum(-1) / (t ** 2).sum()

if TASK == 'cls':      # training dynamics over the EVAL_AT checkpoints
    lg = {ep: torch.load(f'{CKPT_DIR}/{TAG}_out_ep{ep}.pt') for ep in EVAL_AT}
    def per_epoch(logits, labels):
        logp = F.log_softmax(logits.float(), -1); lab = torch.as_tensor(labels)[:, None]
        ce = -logp.gather(1, lab).squeeze(1); ent = -(logp.exp() * logp).sum(1)
        mar = logits.float().gather(1, lab).squeeze(1) - logits.float().scatter(1, lab, float('-inf')).max(1).values
        return ce.numpy(), ent.numpy(), mar.numpy()
    def traj(key, labels):
        ce, ent, mar = (np.stack(z, 1) for z in zip(*[per_epoch(lg[ep][key], labels) for ep in EVAL_AT]))
        return np.concatenate([ce, ent, mar, np.diff(ce, axis=1), np.stack([slope(ce, EVAL_AT), slope(mar, EVAL_AT)], 1)], 1)
    Xf = np.r_[traj('m', pool_y[idx_m]), traj('n', pool_y[idx_n])]
    feat_desc = 'per-epoch CE/entropy/margin + CE deltas + CE/margin slopes'
else:                  # residual shape at the last epoch
    res = np.r_[(X_pool[idx_m][..., LOOKBACK:] - target['m']).numpy(), (X_pool[idx_n][..., LOOKBACK:] - target['n']).numpy()]
    mag = np.abs(np.fft.rfft(res, axis=-1))[..., 1:]
    dft = np.stack([np.log1p(mag[..., b].mean(-1)) for b in np.array_split(np.arange(mag.shape[-1]), 8)], -1).reshape(len(res), -1)
    t_h = np.arange(HORIZON)
    Xf = np.concatenate([dft, slope(res, t_h), slope(np.abs(res), t_h), np.log((res ** 2).mean(-1) + 1e-8)], 1)
    feat_desc = 'log DFT 8 bands + trend + drift + per-lead log MSE of the residual'
yb = np.r_[np.ones(n_m), np.zeros(len(idx_n))]
s = cross_fit(Xf, yb)
save_result('temporal', s[:n_m], s[n_m:], classifier='HistGradientBoosting', n_folds=5, features=feat_desc,
            threat_model='gray-box: labelled member/non-member subset' + (', intermediate checkpoints' if TASK == 'cls' else ''))

## 13. Tóm tắt + ROC log–log

In [ ]:
import matplotlib.pyplot as plt
names = ['loss', 'conf', 'entropy', 'lira', 'lira_offline', 'rmia', 'rmia_online', 'temporal']
fig, ax = plt.subplots(figsize=(6, 6)); tab = []
for a in names:
    p = f'{RESULTS_DIR}/{a}.json'
    if not os.path.exists(p): continue
    r = json.load(open(p)); sm_, sn_ = np.array(r['scores_member']), np.array(r['scores_nonmember'])
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm_)), np.zeros(len(sn_))], np.r_[sm_, sn_])
    ax.plot(fpr, tpr, label=f"{a} (AUC {r['auc']:.3f})")
    tab.append({'attack': a, 'tpr@0.1%': r['tpr_at_0.1fpr'], 'CI': r['bootstrap_ci']['tpr_at_0.1fpr'], 'tpr@1%': r['tpr_at_1fpr'], 'auc': r['auc']})
ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8); ax.set_title(f'{MODEL.upper()} PTB-XL {TASK}, {LEVEL}-level')
plt.savefig(f'{RESULTS_DIR}/roc_loglog.png', dpi=150, bbox_inches='tight'); plt.show()
print(pd.DataFrame(tab).round(4).to_string(index=False)); print('utility', UTIL)

## 14. Hình minh hoạ cho demo
Một member "lộ" nhất và một non-member điển hình: tín hiệu lead II và vị trí φ của target so với 4 shadow IN / 4 shadow OUT.

In [ ]:
def show_record(i_pool, title, ax_sig, ax_phi):
    ax_sig.plot(X_pool[i_pool, 1].numpy(), lw=0.8); ax_sig.set_title(f'{title}: ecg_id {pool_ids[i_pool]}, lead II')
    pin, pout = PHI_S[masks_pool[:, i_pool], i_pool], PHI_S[~masks_pool[:, i_pool], i_pool]
    ax_phi.scatter(pin, np.zeros_like(pin) + 1, label='shadow IN', c='tab:green')
    ax_phi.scatter(pout, np.zeros_like(pout), label='shadow OUT', c='tab:red')
    ax_phi.axvline(PHI_T[i_pool], c='k', ls='--', label='target model')
    ax_phi.set_yticks([0, 1]); ax_phi.set_yticklabels(['OUT', 'IN']); ax_phi.set_xlabel('phi'); ax_phi.legend(fontsize=7)
lira_on = lira(True)
i_mem = idx_m[np.argmax(lira_on[:n_m])]                # most exposed member
i_non = idx_n[np.argmin(np.abs(lira_on[n_m:]))]        # typical non-member
fig, axs = plt.subplots(2, 2, figsize=(12, 5))
show_record(i_mem, 'MEMBER', axs[0, 0], axs[0, 1]); show_record(i_non, 'NON-MEMBER', axs[1, 0], axs[1, 1])
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/demo_lira_example.png', dpi=150); plt.show()

## 15. Đóng gói kết quả
Tải `tv1_results_<model>_<task>.zip` ở tab *Output* (hoặc panel bên phải), giải nén vào `results/<model>/<task>/` của repo rồi commit.

In [ ]:
z = shutil.make_archive(f'{DRIVE_ROOT}/tv1_results_{MODEL}_{TASK}', 'zip', RESULTS_DIR)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB', sorted(os.listdir(RESULTS_DIR)))